In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

In [3]:
from sqlalchemy import create_engine

username = "root"
password = "1779"
host = "localhost"
database = "olist_ecommerce_db"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}/{database}"
)

print("Database Connected Successfully")

Database Connected Successfully


In [4]:
customers = pd.read_sql("SELECT * FROM customers", engine)
orders = pd.read_sql("SELECT * FROM orders", engine)
payments = pd.read_sql("SELECT * FROM payments", engine)
order_items = pd.read_sql("SELECT * FROM order_items", engine)
products = pd.read_sql("SELECT * FROM products", engine)
reviews = pd.read_sql("SELECT * FROM order_reviews", engine)
sellers = pd.read_sql("SELECT * FROM sellers", engine)
category = pd.read_sql("SELECT * FROM category_translation", engine)

In [5]:
master_df = orders.merge(customers, on="customer_id", how="left")

master_df = master_df.merge(payments, on="order_id", how="left")

master_df = master_df.merge(reviews, on="order_id", how="left")

master_df = master_df.merge(order_items, on="order_id", how="left")

master_df = master_df.merge(products, on="product_id", how="left")

master_df = master_df.merge(category, on="product_category_name", how="left")

master_df = master_df.merge(sellers, on="seller_id", how="left")

print(master_df.shape)

(119143, 40)


In [ ]:
#    Feature Engineering

# delivery_days	
# order_month	
# purchase_hour
# is_late_delivery
# order_value_bucket
# freight_percentage
# review_category

In [ ]:
#Delivery Days

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    master_df[col] = pd.to_datetime(master_df[col])

In [7]:
master_df.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
customer_unique_id                          str
customer_zip_code_prefix                  int64
customer_city                               str
customer_state                              str
payment_sequential                      float64
payment_type                                str
payment_installments                    float64
payment_value                           float64
review_id                                   str
review_score                            float64
review_comment_title                        str
review_comment_message                      str
review_creation_date                    

In [8]:
master_df.shape

(119143, 40)

In [9]:
master_df["delivery_days"] = (
    master_df["order_delivered_customer_date"]
    -
    master_df["order_purchase_timestamp"]
).dt.days

In [10]:
master_df[
[
"order_purchase_timestamp",
"order_delivered_customer_date",
"delivery_days"
]
].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
2,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
3,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
4,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0


In [11]:
master_df[
[
"order_purchase_timestamp",
"order_delivered_customer_date",
"delivery_days"
]
].head()

,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
1,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
2,2017-10-02 10:56:33,2017-10-10 21:25:13,8.0
3,2018-07-24 20:41:37,2018-08-07 15:27:45,13.0
4,2018-08-08 08:38:49,2018-08-17 18:06:29,9.0


In [12]:
#Order Month
master_df["order_month"] = (
    master_df["order_purchase_timestamp"]
    .dt.month_name()
)

In [13]:
master_df[
[
"order_purchase_timestamp",
"order_month"
]
].head()

,order_purchase_timestamp,order_month
0,2017-10-02 10:56:33,October
1,2017-10-02 10:56:33,October
2,2017-10-02 10:56:33,October
3,2018-07-24 20:41:37,July
4,2018-08-08 08:38:49,August


In [14]:
#Order Year
master_df["order_year"] = (
    master_df["order_purchase_timestamp"]
    .dt.year
)

In [15]:
master_df[
[
"order_purchase_timestamp",
"order_year"
]
].head()

,order_purchase_timestamp,order_year
0,2017-10-02 10:56:33,2017
1,2017-10-02 10:56:33,2017
2,2017-10-02 10:56:33,2017
3,2018-07-24 20:41:37,2018
4,2018-08-08 08:38:49,2018


In [16]:
#Purchase Hour
master_df["purchase_hour"] = (
    master_df["order_purchase_timestamp"]
    .dt.hour
)

In [17]:
master_df[
[
"order_purchase_timestamp",
"purchase_hour"
]
].head()

,order_purchase_timestamp,purchase_hour
0,2017-10-02 10:56:33,10
1,2017-10-02 10:56:33,10
2,2017-10-02 10:56:33,10
3,2018-07-24 20:41:37,20
4,2018-08-08 08:38:49,8


In [18]:
master_df.shape

(119143, 44)

In [19]:
#is_late_delivery
master_df["is_late_delivery"] = (
    master_df["order_delivered_customer_date"] >
    master_df["order_estimated_delivery_date"]
)

In [20]:
master_df[
[
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "is_late_delivery"
]
].head()

,order_delivered_customer_date,order_estimated_delivery_date,is_late_delivery
0,2017-10-10 21:25:13,2017-10-18,False
1,2017-10-10 21:25:13,2017-10-18,False
2,2017-10-10 21:25:13,2017-10-18,False
3,2018-08-07 15:27:45,2018-08-13,False
4,2018-08-17 18:06:29,2018-09-04,False


In [21]:
#order_value_bucket
master_df["order_value_bucket"] = pd.cut(
    master_df["payment_value"],
    bins=[0,100,500,1000,100000],
    labels=["Low","Medium","High","Premium"]
)

In [22]:
master_df[
[
    "payment_value",
    "order_value_bucket"
]
].head(10)

,payment_value,order_value_bucket
0,18.12,Low
1,2.00,Low
2,18.59,Low
3,141.46,Medium
4,179.12,Medium
5,72.20,Low
6,28.62,Low
7,175.26,Medium
8,65.95,Low
9,75.16,Low


In [23]:
#freight_percentage
master_df["freight_percentage"] = (
    master_df["freight_value"] /
    master_df["payment_value"]
) * 100

In [24]:
master_df[
[
    "payment_value",
    "freight_value",
    "freight_percentage"
]
].head()

,payment_value,freight_value,freight_percentage
0,18.12,8.72,48.123620
1,2.00,8.72,436.000000
2,18.59,8.72,46.906939
3,141.46,22.76,16.089354
4,179.12,19.22,10.730237


In [25]:
#review_category
master_df["review_category"] = np.where(
    master_df["review_score"] >= 4,
    "Positive",
    np.where(
        master_df["review_score"] == 3,
        "Neutral",
        "Negative"
    )
)

In [26]:
master_df[
[
    "review_score",
    "review_category"
]
].head(10)

,review_score,review_category
0,4.0,Positive
1,4.0,Positive
2,4.0,Positive
3,4.0,Positive
4,5.0,Positive
5,5.0,Positive
6,5.0,Positive
7,4.0,Positive
8,2.0,Negative
9,5.0,Positive


In [27]:
master_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'review_id', 'review_score',
       'review_comment_title', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp', 'order_item_id',
       'product_id', 'seller_id', 'shipping_limit_date', 'price',
       'freight_value', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english', 'seller_zip_code_prefix',
       'seller_city', 'seller_state', 'delivery_days', 'order_month',
       'order_year', 

In [28]:
master_df.shape

(119143, 48)